# 순환 신경망 기반 감성 분석 (네이버 영화 리뷰)
1. 데이터 준비
2. 모델 구축 및 컴파일
3. 모델 학습
4. 모델 평가
5. 예측
6. 배포 (저장, 재사용)

## 1. 데이터 준비
    1-1. 데이터 로딩
    1-2. 데이터 전처리
    1-3. 데이터 분리 
    1-4. 학습용 데이터 준비
    1-5. 테스트용 데이터준비 

### 1-1. 데이터 로딩

In [1]:
# naver_movie_review.csv
import pandas as pd
datafile = '../data/naver_movie_review.csv'
review_df = pd.read_csv(datafile)
review_df.head()


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [2]:
review_df.label.value_counts()

label
0    100000
1    100000
Name: count, dtype: int64

In [3]:
review_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        200000 non-null  int64 
 1   document  199992 non-null  object
 2   label     200000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.6+ MB


### 1-2. 데이터 전처리
- 결측치 제거
- 정제 (한글만 남기고 모두 삭제)
- 중복치 제거
- 형태소 분석기로 토큰화

#### 1-2-1. 결측치 제거

In [4]:
# 결측치 확인
review_df.isnull().sum()

id          0
document    8
label       0
dtype: int64

In [5]:
review_df[review_df.document.isnull()]

,id,document,label
25857,2172111,NaN,1
55737,6369843,NaN,1
110014,1034280,NaN,0
126782,5942978,NaN,0
140721,1034283,NaN,0
155746,402110,NaN,1
157899,5026896,NaN,0
177097,511097,NaN,1


In [6]:
# 결측치 제거
review_df.dropna(inplace=True)
review_df.isnull().sum()

id          0
document    0
label       0
dtype: int64

#### 1-2-2. 정제
* 한글과 공백을 제외한 문자는 공백으로 치환하여 제거
* 한글이 없었던 문장은 공백만 남아있게 되므로, 결측치 삭제 처리 진행 필요

In [7]:
import re
# 한글과 공백 제외하고 모두 공백으로 치환
review_df['clean_review'] = review_df.document.apply(lambda x : re.sub('[^ 가-힣]+', ' ', x))
# 문장의 시작 부분에 있는 공백을 ""으로 치환 
review_df.clean_review = review_df.clean_review.apply(lambda x : re.sub('^ +', '', x))

# 빈 문자열("")은 결측치로 수정
review_df.clean_review = review_df.clean_review.replace('', None)

review_df.head()

,id,document,label,clean_review
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0,아 더빙 진짜 짜증나네요 목소리
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1,흠 포스터보고 초딩영화줄 오버연기조차 가볍지 않구나
2,10265843,너무재밓었다그래서보는것을추천한다,0,너무재밓었다그래서보는것을추천한다
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0,교도소 이야기구먼 솔직히 재미는 없다 평점 조정
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...


In [8]:
# 결측치 확인
review_df.clean_review.isnull().sum()

np.int64(2177)

In [9]:
# 결측치 제거
review_df.dropna(subset=['clean_review'], inplace=True)

In [10]:
review_df.clean_review.isnull().sum()

np.int64(0)

#### 1-2-3. 중복치 제거

In [11]:
# 중복치 확인
review_df.clean_review.duplicated().sum()

np.int64(6550)

In [12]:
review_df[review_df.clean_review.duplicated(keep=False)].sort_values(by=['clean_review'])

,id,document,label,clean_review
81436,8522472,가나다라마바사아자차,0,가나다라마바사아자차
37314,9593066,가나다라마바사아자차,0,가나다라마바사아자차
169301,10230088,가나다라마바사아자차,1,가나다라마바사아자차
9137,9629024,가나다라마바사아자차,1,가나다라마바사아자차
154484,8145963,가나다라마바사아자차,0,가나다라마바사아자차
...,...,...,...,...
8446,5158304,힐러리 더프의 매력에 빠지다!!!,1,힐러리 더프의 매력에 빠지다
167827,4052413,힘내세요,0,힘내세요
134993,3010576,힘내세요,0,힘내세요
188788,7024515,힘들다,0,힘들다


In [28]:
# 중복치 제거
review_df.drop_duplicates(subset=['clean_review'], inplace=True)
review_df.clean_review.duplicated().sum()


np.int64(0)

In [29]:
review_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 191265 entries, 0 to 199999
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            191265 non-null  int64 
 1   document      191265 non-null  object
 2   label         191265 non-null  int64 
 3   clean_review  191265 non-null  object
dtypes: int64(2), object(2)
memory usage: 7.3+ MB


#### 1-2-4. 토큰화

In [42]:
# 형태소 분석기 적용 (Okt, Komoran 등)
from konlpy.tag import Okt
from tqdm import tqdm

tqdm.pandas()

review_df['tokens'] = review_df.clean_review.progress_apply(Okt().morphs) # prograss 하면 tqdm() 한거랑 같은 효과 인듯? 


In [43]:
review_df['tokens_str'] = review_df.tokens.apply(lambda x : ' '.join(x))
review_df.head()

,id,document,label,clean_review,tokens,tokens_str
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0,아 더빙 진짜 짜증나네요 목소리,"[아, 더빙, 진짜, 짜증나네요, 목소리]",아 더빙 진짜 짜증나네요 목소리
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1,흠 포스터보고 초딩영화줄 오버연기조차 가볍지 않구나,"[흠, 포스터, 보고, 초딩, 영화, 줄, 오버, 연기, 조차, 가볍지, 않구나]",흠 포스터 보고 초딩 영화 줄 오버 연기 조차 가볍지 않구나
2,10265843,너무재밓었다그래서보는것을추천한다,0,너무재밓었다그래서보는것을추천한다,"[너, 무재, 밓었, 다그, 래서, 보는것을, 추천, 한, 다]",너 무재 밓었 다그 래서 보는것을 추천 한 다
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0,교도소 이야기구먼 솔직히 재미는 없다 평점 조정,"[교도소, 이야기, 구먼, 솔직히, 재미, 는, 없다, 평점, 조정]",교도소 이야기 구먼 솔직히 재미 는 없다 평점 조정
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,"[사이, 몬페, 그, 의, 익살스런, 연기, 가, 돋보였던, 영화, 스파이더맨, 에...",사이 몬페 그 의 익살스런 연기 가 돋보였던 영화 스파이더맨 에서 늙어 보이기만 했...


In [44]:
review_df.to_csv(datafile.replace('.csv', '_ing.csv'))

In [41]:
from tqdm import tqdm
test_list = [1] * 100000000
for test in tqdm(test_list):
    test += 1

100%|██████████| 100000000/100000000 [00:09<00:00, 10858454.74it/s]


### 1-3. 데이터 분리
* 정답 데이터의 분포 확인 -> 학습용 데이터와 테스트 데이터로 분리 시 비율 유지 (stratify=y)

In [15]:
# label 컬럼의 값별 데이터 수 확인


In [16]:
# 막대그래프로 그려보기


In [17]:
# 입력 데이터와 정답데이터 추출  (list)


In [18]:
# 학습 데이터와 테스트 데이터 분리


### 1-4. 학습 데이터 준비
    1-4-1. Integer Encoding을 위한 tokenizer 생성
    1-4-2. 입력 데이터 Integer Encoding
    1-4-3. 입력 데이터 Padding
    1-4-4. 정답 데이터 원핫인코딩

#### 1-4-1. Integer Encoding을 위한 tokenizer 생성
* num_words = 사용할 단어 수(vocab_size) + 1 (0은 OOV에 할당)

In [19]:
# 단어 수 제한없이 Tokenizer 생성하여 단어 수 확인


In [20]:
# 등장 빈도수를 threshold로 설정하여 버릴 단어가 차지하는 비율 확인


In [21]:
# 단어 수를 제한하여 tokenizer 생성


#### 1-4-2. 입력 데이터 Integer Encoding
* 제한된 단어에만 index를 부여하므로, 희귀 단어로만 구성된 review는 단어가 0이 되므로 결측치에 해당 -> 결측치 제거

In [22]:
# 입력 데이터 Integer Encoding


In [23]:
# 길이가 0인 리뷰의 index 추출


In [24]:
# 길이가 0인 리뷰가 있는 경우, 길이가 1 이상인 리뷰로 학습데이터 재구성


#### 1-4-3. 입력 데이터 padding
* 입력 데이터의 길이(max_len)를 정하여 padding

In [25]:
# 리뷰 길이 분포 확인 (히스토그램 그려보기)


In [26]:
# 최대, 최소, 평균 등 정보 확인


In [27]:
# 길이가 max_len 이하인 데이터의 비중 확인
max_len = 

SyntaxError: invalid syntax (4236980498.py, line 2)

In [ ]:
# max_len 길이로 입력 데이터 padding


#### 1-4-4. 정답 데이터 one-hot encoding

### 1-5. 테스트 데이터 준비
    1-5-1. 입력 데이터 Integer Encoding (결측치제거) 
    1-5-2. 입력 데이터 padding
    1-5-3. 정답 데이터 one-hot encoding

In [ ]:
# 입력 데이터 Integer Encoding


In [ ]:
# 길이가 0인 리뷰의 index 추출 -> 있으면 길이가 1 이상인 리뷰만으로 테스트 데이터 재구성 


In [ ]:
# 입력 데이터 padding


In [ ]:
# 정답 데이터 one-hot encoding


## 2. 모델 구축 및 컴파일

In [ ]:
# 모델 설계


In [ ]:
# 모델 생성


In [ ]:
# 모델 컴파일


## 3. 모델 학습

In [ ]:
# EarlyStopping, ModelCheckpoint callback 함수 설정


In [ ]:
# 모델 학습


## 4. 모델 평가

In [ ]:
# 저장된 모델 로딩하여, 테스트 데이터로 평가


In [ ]:
# predict로 테스트 데이터의 예측값 구기기


In [ ]:
# sklearn의 classification_report()로 평가 결과 확인


## 5. 예측

In [ ]:
# 입력된 리뷰에 대한 긍부정 판단 함수


In [ ]:
# 함수 테스트
reviews = [
    '이 영화 개꿀잼 ㅋㅋㅋ',
    '하품만 나온다',
    '이 영화 핵노잼 ㅠㅠ',
    '이딴게 영화냐 ㅉㅉ',
    '와 개쩐다',
    '감독 뭐하는 놈이냐',
    '정말 세계관 최강자들의 영화다'
]



## 6. 배포 (모델 저장 및 재사용)
    6-1. 모델 저장
    6-2. SentimentAnalyzer 클래스 구현
        *  저장된 모델 로딩 및 사용

### 6-1. 모델 저장 

In [ ]:
# keras 학습 모델저장 


In [ ]:
# Integer Encoding을 위한 파이썬 객체 직렬화 (encoder)


### 6-2. SentimentAnalyzer 클래스 구현
- 객체 생성 시 예측 모델, Integer Encoder 로딩
- 한국어 형태소 분석기 정의
- 입력된 리뷰의 긍부정 판단 함수